# การวิเคราะห์ปริมาณการเดินทางและผู้โดยสาร (ชุดข้อมูลที่ 4)
**รายวิชา:** AIE314 Data Preparation and Data Mining 
**กลุ่ม:** Loading Brain... 1%
1. ธีรนัย ธีรภาพวงศา 1670903515
2. ธนภัค อัครธนาวัสส์ 1670904190
3. กัญญพัชร รอดคล้าย 1670902111

# 1. Data Loading & Data Understanding
*ขั้นนี้อ่านข้อมูลดิบทั้งสองไฟล์ ทำความเข้าใจว่า 1 แถวคืออะไร และอธิบายความหมายของแต่ละคอลัมน์ เพื่อเลือกคอลัมน์ที่ใช้ตอบโจทย์ทั้ง 3 ข้อ ยังไม่แก้ข้อมูล (AIE315 Lab 1: Data Profiling, AIE315 W8: Domain Audit และ Redundancy)*

## 1.1 อ่านไฟล์และดูหน้าตาข้อมูล

In [9]:
import pandas as pd

DATA_DIR = '../data/'

df68_raw = pd.read_csv(DATA_DIR + "4_passengers68.csv", dtype=str)
df69_raw = pd.read_csv(DATA_DIR + "4_passengers69.csv", dtype=str)

print("ขนาดข้อมูลของปี 2568: ", df68_raw.shape)
print("ขนาดข้อมูลของปี 2569: ", df69_raw.shape)

df68_raw.head()
# df69_raw.head()

 

ขนาดข้อมูลของปี 2568:  (69440, 8)
ขนาดข้อมูลของปี 2569:  (3010, 8)


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
0,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,บขส.,รถ บขส. และ รถร่วม,01/01/2025,คน,"127,551"
1,ทางถนน,การเดินทางระหว่างจังหวัด,สาธารณะ,ขบ.,รถหมวด 3,01/01/2025,คน,"8,218"
2,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์เฉพาะ 4 ล้อ (10 จุดสำรวจ),01/01/2025,คัน,"877,943"
3,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,ทล.,รถยนต์ทุกประเภท (10 จุดสำรวจ),01/01/2025,คัน,"932,642"
4,ทางถนน,การเดินทางระหว่างจังหวัด,ส่วนบุคคล,กทพ.,รถยนต์เฉพาะ 4 ล้อ (ทางด่วน),01/01/2025,คัน,"1,364,992"


## 1.2 ทำความเข้าใจว่า 1 แถวคืออะไร

**ถ้า 1 แถว = 1 บริการ ใน 1 วัน จำนวนแถวต้องเท่ากับ จำนวนบริการ × จำนวนวัน พอดี ถ้าไม่เท่า แปลว่ามีแถวเกินหรือขาด และต้องหาสาเหตุ**

*มี 43 บริการ แบ่งเป็น 4 รูปแบบการเดินทาง และ 3 วัตถุประสงค์*

*ปี 68 ครอบคลุม 365 วัน ปี 69 มีแค่ 70 วัน (1 ม.ค.–11 มี.ค. 2026)*
 
1. ในกรณีนี้ ปี68: 43 × 365 = 15,695 แต่ไฟล์มี 69,440 แถว 
  - แถวที่ไม่ว่างมี 14,236 + 1,460 = 15,696 แถว
  - ส่วน 53,744 แถว ว่างทุกช่อง อยู่ท้ายไฟล์ เริ่มตั้งแต่ index 15,696 เป็นต้นไป
  - 15,696 − 15,695 = เกินมา 1 แถว แต่ยังยืนยันไม่ได้ (ตรวจใน 2.1)
2. ในกรณีนี้ ปี69: 43 × 70 = 3,010 ตรงกับ .shape พอดี แปลว่า 1 แถว = 1 บริการ × 1 วัน 
   
   

In [10]:
print("-- unique values 2568 -- \n", df68_raw.nunique())
print("-" * 30)
print("-- unique values 2569 -- \n", df69_raw.nunique())
print("-" * 30)

print("นับหน่วยของปี 2568", df68_raw["หน่วย"].value_counts(dropna= False))
# print("นับหน่วยของปี 2569", df69_raw["หน่วย"].value_counts(dropna= False))
print("-" * 30)

df68_raw.tail()

-- unique values 2568 -- 
 รูปแบบการเดินทาง         4
วัตถุประสงค์             3
สาธารณะ/ส่วนบุคคล        2
หน่วยงาน                13
ยานพาหนะ/ท่า            43
วันที่                 365
หน่วย                    2
ปริมาณ               12162
dtype: int64
------------------------------
-- unique values 2569 -- 
 รูปแบบการเดินทาง        4
วัตถุประสงค์            3
สาธารณะ/ส่วนบุคคล       2
หน่วยงาน               13
ยานพาหนะ/ท่า           43
วันที่                 70
หน่วย                   2
ปริมาณ               2528
dtype: int64
------------------------------
นับหน่วยของปี 2568 หน่วย
NaN    53744
คน     14236
คัน     1460
Name: count, dtype: int64
------------------------------


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,วันที่,หน่วย,ปริมาณ
69435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69436,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
69439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 1.3 Data Dictionary และ Feature Selection

*Data Dictionary บอกว่าแต่ละคอลัมน์หมายถึงอะไร เป็นข้อมูลประเภทไหน และมีค่าอะไรบ้าง*

*Feature Selection ในโครงงานนี้ไม่ใช้วิธีทางสถิติแบบ W8 เพราะเราไม่ได้สร้างโมเดล แต่ใช้หลัก Domain Audit และ Redundancy ของ W8 คือ ถ้าสองคอลัมน์บอกข้อมูลเดียวกัน ให้เก็บตัวแทนไว้ตัวเดียว*


In [11]:
desc_cols = ["รูปแบบการเดินทาง", "วัตถุประสงค์", "สาธารณะ/ส่วนบุคคล", "หน่วยงาน", "ยานพาหนะ/ท่า", "หน่วย"]

service_list_68 = df68_raw[desc_cols].drop_duplicates().dropna(how= "all")
print(f"จำนวนการบริการทั้งหมด (ไม่ซ้ำ, ไม่มีค่าว่างทั้งเเถว): {len(service_list_68)} เเถว")
print("-" * 100)
# service_list_69 = df69_raw[desc_cols].drop_duplicates().dropna(how= "all")
# print(f"จำนวนการบริการทั้งหมด (ไม่ซ้ำ, ไม่มีค่าว่างทั้งเเถว): {len(service_list_69)} เเถว")
# print("-" * 100)

service_list_68.sort_values(["วัตถุประสงค์", "รูปแบบการเดินทาง"])

จำนวนการบริการทั้งหมด (ไม่ซ้ำ, ไม่มีค่าว่างทั้งเเถว): 43 เเถว
----------------------------------------------------------------------------------------------------


,รูปแบบการเดินทาง,วัตถุประสงค์,สาธารณะ/ส่วนบุคคล,หน่วยงาน,ยานพาหนะ/ท่า,หน่วย
6,ทางถนน,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,ขบ.,รถหมวด 4,คน
7,ทางถนน,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,ขสมก.,รถเมล์ ขสมก.,คน
8,ทางถนน,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,ขสมก.,รถร่วม (หมวด 1),คน
9,ทางถนน,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,ขบ.,รถเอกชนเส้นปฏิรูป (หมวด 1),คน
10,ทางน้ำ,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,จท.,เรือด่วนเจ้าพระยา,คน
11,ทางน้ำ,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,จท.,เรือคลองแสนแสบ,คน
12,ทางน้ำ,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,จท.,เรือข้ามฟากเจ้าพระยา,คน
13,ทางน้ำ,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,จท.,เรือไฟฟ้าเจ้าพระยา,คน
14,ทางน้ำ,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,จท.,เรือภูมิภาค,คน
16,ทางราง,การเดินทางภายในจังหวัด/กรุงเทพ,สาธารณะ,รฟม.,รถไฟฟ้าสายสีน้ำเงิน,คน


In [12]:
# จำนวนบริการแยกตามวัตถุประสงค์ (หลักฐานของตัวเลข 43 บริการใน 1.2)
service_by_purpose = service_list_68.groupby("วัตถุประสงค์").size()
display(service_by_purpose)
print("รวม: ", service_by_purpose.sum(), "บริการ")

วัตถุประสงค์
การเดินทางภายในจังหวัด/กรุงเทพ    16
การเดินทางระหว่างจังหวัด          13
การเดินทางระหว่างประเทศ           14
dtype: int64

รวม:  43 บริการ


### Data Dictionary

| คอลัมน์ | ความหมาย | ประเภทข้อมูล (AIE314 บทที่ 2) | ค่าที่พบ |
| :--- | :--- | :---: | :--- |
| รูปแบบการเดินทาง | ช่องทางการเดินทาง | Nominal | 4 ค่า: ถนน, ราง, น้ำ, อากาศ |
| วัตถุประสงค์ | ขอบเขตการเดินทาง | Nominal | 3 ค่า: ภายในจังหวัด/กทม., ระหว่างจังหวัด, ระหว่างประเทศ |
| สาธารณะ/ส่วนบุคคล | ประเภทผู้ให้บริการ | Nominal | 2 ค่า |
| หน่วยงาน | หน่วยงานที่รายงานตัวเลข | Nominal | 13 ค่า |
| ยานพาหนะ/ท่า | ชื่อบริการหรือท่า | Nominal | 43 ค่า |
| วันที่ | วันที่ของข้อมูล (dd/mm/ค.ศ.) | วันที่ (เก็บเป็นข้อความ) | 365 วัน / 70 วัน |
| หน่วย | หน่วยของปริมาณ | Nominal | คน, คัน |
| ปริมาณ | จำนวนผู้ใช้บริการหรือจำนวนรถต่อวัน | Discrete (ยังเป็นข้อความที่มี `,`) | 0 ถึงหลักล้าน, มีค่าว่าง |

### Feature Selection

| คอลัมน์ | ข้อ 1 | ข้อ 2 | ข้อ 3 | เหตุผล |
| :--- | :---: | :---: | :---: | :--- |
| วันที่ | ใช้ข้อมูล | ใช้ข้อมูล | ใช้ข้อมูล | ข้อ 1 ใช้สร้างวันในสัปดาห์และเดือน, ข้อ 2 ใช้เลือกช่วงวันที่เทียบกัน, ข้อ 3 ใช้รวมเป็นรายเดือน |
| ยานพาหนะ/ท่า | ใช้ข้อมูล | ใช้ข้อมูล | ใช้ข้อมูล | ตัวระบุบริการ ใช้ `groupby()` และสร้างทิศทางขาเข้า/ขาออกสำหรับข้อ 3 |
| ปริมาณ | ใช้ข้อมูล | ใช้ข้อมูล | ใช้ข้อมูล | ค่าที่ใช้วิเคราะห์ |
| หน่วย | ใช้ข้อมูล | ใช้ข้อมูล | ใช้ข้อมูล | ใช้แยกคนกับคัน ตามที่โจทย์ห้ามรวม |
| รูปแบบการเดินทาง | ใช้ข้อมูล | ใช้ข้อมูล | ใช้ข้อมูล | ใช้สรุปเป็นกลุ่ม และข้อ 3 กำหนดให้จำแนกตามรูปแบบ |
| วัตถุประสงค์ | ใช้ประกอบ | ใช้ประกอบ | ใช้ข้อมูล | ข้อ 3 ใช้กรองเฉพาะ "ระหว่างประเทศ" |
| หน่วยงาน | <u>ไม่ใช้ข้อมูล</u> | <u>ไม่ใช้ข้อมูล</u> | <u>ไม่ใช้ข้อมูล</u> | ใช้ตรวจสอบการนับซ้อน ไม่ใช้วิเคราะห์โดยตรง |
| สาธารณะ/ส่วนบุคคล | <u>ไม่ใช้ข้อมูล</u> | <u>ไม่ใช้ข้อมูล</u> | <u>ไม่ใช้ข้อมูล</u> | ซ้ำซ้อนกับ `หน่วย` 100% |

*หมายเหตุ: วันที่ และ ปริมาณ ยังเก็บเป็นข้อความ จะแปลงเป็น datetime และตัวเลขในขั้นที่ 3*